<a href="https://colab.research.google.com/github/diego-ascenciorod/Clase-Laboratorio-Estadistico/blob/main/A06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# instalar libreria
!pip install scikit-optimize -q

# librerias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, confusion_matrix, roc_curve, auc
from skopt import BayesSearchCV
from skopt.space import Real

# leer archivo
df = pd.read_csv("brain_tumor_dataset.csv.zip")

# variable objetivo
y_col = "Tumor_Type"


print("primeras filas")
print(df.head())

print("\ninfo")
print(df.info())

print("\nnulos")
print(df.isnull().sum())

print("\nconteo de la variable objetivo")
print(df[y_col].value_counts())

# grafica de la variable objetivo
plt.figure(figsize=(6,4))
sns.countplot(x=y_col, data=df)
plt.title("distribucion de tumor type")
plt.xlabel("tipo")
plt.ylabel("frecuencia")
plt.show()

# separar x y y
X = df.drop(["Patient_ID", y_col], axis=1)
y = df[y_col]

# pasar texto a numeros
X = pd.get_dummies(X, drop_first=True)

le = LabelEncoder()
y = le.fit_transform(y)

print("\ncolumnas finales de x")
print(X.columns)

print("\nshape de x:", X.shape)
print("shape de y:", y.shape)

# dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# escalar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# modelo
modelo = SVC(
    kernel="linear",
    probability=True,
    random_state=42
)

# validacion cruzada
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# optimizacion bayesiana
opt = BayesSearchCV(
    estimator=modelo,
    search_spaces={
        "C": Real(1e-3, 1e+1, prior="log-uniform")
    },
    n_iter=5,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
    random_state=42
)

print("\nse esta entrenando el modelo...")
opt.fit(X_train, y_train)

# mejor modelo
mejor_modelo = opt.best_estimator_

# predicciones
y_pred = mejor_modelo.predict(X_test)
y_prob = mejor_modelo.predict_proba(X_test)[:, 1]

# precision y recall
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

# f1 con la formula
if precision + recall == 0:
    f1 = 0
else:
    f1 = 2 * (precision * recall) / (precision + recall)

# auc
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# resultados
print("\n" + "="*40)
print("resultados finales")
print("="*40)
print("mejor C:", round(opt.best_params_["C"], 4))
print("mejor auc cv:", round(opt.best_score_, 4))
print("precision:", round(precision, 4))
print("recall:", round(recall, 4))
print("f1:", round(f1, 4))
print("auc test:", round(roc_auc, 4))
print("="*40)

# matriz de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("matriz de confusion")
plt.xlabel("predicho")
plt.ylabel("real")
plt.show()

# curva roc
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("falsos positivos")
plt.ylabel("verdaderos positivos")
plt.title("curva roc")
plt.legend()
plt.show()